# Phase 1: Data Preprocessing
**Project:** Comparative Opinion Mining for Arabic Social Media  
**Goal:** Clean, normalize, and prepare Arabic tweet data for comparative sentence detection.

### Fixes applied vs original notebook
1. Removed `text.lower()` from Arabic cleaning — harmful on Arabic characters  
2. `remove_diacritics` runs **before** any regex — prevents diacritics breaking patterns  
3. Brand variants stored in **normalized form** — إسوس/أسوس → اسوس — so they match normalized text  
4. Added missing brands: `xbox` (under microsoft), `htc`  
5. **CRITICAL**: `Com_df` contains 504 rows labeled `non` — fixed to assign label by column value, not hard-coded `1`  
6. `protect_brands` handles multi-word names correctly  
7. Product qualifiers (`ultra`, `pro`, `max`, `mini`...) preserved when removing English  
8. Label-noise filter threshold corrected (`brand_count >= 1`, not `>= 2`)  

## 1. Import Libraries

In [ ]:
# Standard libraries — no heavy dependencies needed for preprocessing
import pandas as pd
import numpy as np
import re
import os
from collections import Counter

## 2. Load Datasets

In [ ]:
Non_df = pd.read_csv('non Com.csv')
Com_df = pd.read_excel('idx_data.xlsx')

print(f'Non-comparative rows : {len(Non_df)}')
print(f'Comparative rows     : {len(Com_df)}')

## 3. Exploratory Data Analysis

In [ ]:
Non_df.head()

In [ ]:
Com_df.head()

In [ ]:
Non_df.info()

In [ ]:
Com_df.info()

In [ ]:
Non_df.isna().sum()

In [ ]:
Com_df.isna().sum()

In [ ]:
# Quick label distribution check on the comparative file
# Raw labels often have trailing spaces — important to catch early
print(Com_df['Compartive'].value_counts(dropna=False))

### EDA Findings

1. Non-effective features in both datasets: `Tweet_id`, `Date`, `User Location`, and `idx`  
2. Missing values in both datasets, especially in `EN1` (493) and `EN2` (1011)  
3. Non-comparative data contains tweets with comparison keywords (label noise)  
4. Comparative data has tweets with missing entities (cannot perform NER)  
5. Text contains English letters, emojis, mentions, links, and hashtags  
6. Extremely short (<3 words) and long (>50 words) tweets exist  
7. Brand names appear in multiple variations (Apple / ابل / ايفون)  
8. Duplicated tweets across the dataset  
9. Arabic diacritics (تشكيل) cause the same word to tokenize differently  
10. Raw labels have trailing/leading whitespace (`'com '`, `' non'`)  

## 4. Text Preprocessing

In [ ]:
# Arabic diacritics (tashkeel) Unicode ranges
# FIX: diacritics removal is defined first so it can be called
# BEFORE any regex operations — diacritics used to silently break
# character-level patterns like Alef normalization.
DIACRITICS = re.compile(
    r'[\u0610-\u061A\u064B-\u065F\u0670'
    r'\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]'
)

def remove_diacritics(text):
    """Remove Arabic tashkeel so the same word isn't tokenized differently."""
    return DIACRITICS.sub('', text)


def normalize_arabic(text):
    """Normalize Arabic character variants to a single canonical form."""
    # Unify all Alef forms into bare Alef
    text = re.sub('[إأآاٱ]', 'ا', text)
    # Alef Maqsura → Ya
    text = re.sub('ى', 'ي', text)
    # Waw Hamza / Ya Hamza → bare Hamza
    text = re.sub('ؤ', 'ء', text)
    text = re.sub('ئ', 'ء', text)
    # Ta Marbuta → Ha
    text = re.sub('ة', 'ه', text)
    # Remove tatweel (elongation dashes used for styling)
    text = re.sub('ـ', '', text)
    return text


def clean_text(text):
    """
    Clean a raw Arabic tweet.

    FIX: Removed .lower() — it has no effect on Arabic characters
    and can corrupt mixed-script tokens (e.g. brand names).

    FIX: remove_diacritics now runs FIRST so diacritics do not
    interfere with subsequent regex patterns.
    """
    text = str(text)

    # Step 1 — strip diacritics first so regexes below work cleanly
    text = remove_diacritics(text)

    # Step 2 — remove @mentions (not useful for content analysis)
    text = re.sub(r'@\w+', '', text)

    # Step 3 — remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Step 4 — remove hashtag symbol but keep the word after it
    text = re.sub(r'#', '', text)

    # Step 5 — remove emojis (broad Unicode supplementary range)
    text = re.sub(r'[\U00010000-\U0010ffff]', ' ', text)

    # Step 6 — remove punctuation (Arabic and Latin)
    text = re.sub(r'[!?.,،؟؛:\-_"()؛،]', ' ', text)

    # Step 7 — normalize Arabic characters after the text is clean
    text = normalize_arabic(text)

    # Step 8 — collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def preprocess(text):
    """Full preprocessing pipeline: clean → normalize."""
    return clean_text(text)

## 5. Brand Normalization

In [ ]:
# Map canonical brand names to all known spelling variants
# (Arabic, English, product sub-brands)
#
# FIX: brand variants are now stored in their NORMALIZED form (post normalize_arabic)
# because extract_brands runs on already-normalized text.
# e.g. 'إسوس' and 'أسوس' both become 'اسوس' after normalize_arabic,
# so we store 'اسوس' as the variant — not the raw form.
#
# FIX: Added missing brands found in data: 'xbox', 'htc', 'msi variants'
brand_map = {
    'apple':     ['ابل', 'apple', 'ايفون', 'iphone', 'ipad', 'mac', 'ios'],
    'samsung':   ['سامسونج', 'samsung', 'جالكسي', 'galaxy'],
    'sony':      ['سوني', 'sony', 'بلايستيشن', 'playstation', 'ps4', 'ps5'],
    'huawei':    ['هواوي', 'huawei'],
    'xiaomi':    ['شاومي', 'xiaomi', 'redmi', 'poco'],
    'microsoft': ['مايكروسوفت', 'microsoft', 'surface', 'xbox'],  # xbox = Microsoft
    'lg':        ['الجي', 'lg'],
    'dell':      ['ديل', 'dell'],
    'hp':        ['اتش بي', 'hp'],
    'lenovo':    ['لينوفو', 'lenovo'],
    'asus':      ['اسوس', 'asus'],   # FIX: إسوس/أسوس → اسوس after normalize_arabic
    'acer':      ['ايسر', 'acer'],
    'nokia':     ['نوكيا', 'nokia'],
    'oneplus':   ['ون بلس', 'oneplus'],
    'oppo':      ['اوبو', 'oppo'],
    'vivo':      ['فيفو', 'vivo'],
    'realme':    ['ريلمي', 'realme'],
    'honor':     ['هونر', 'honor'],
    'google':    ['جوجل', 'قوقل', 'google', 'pixel'],
    'toshiba':   ['توشيبا', 'toshiba'],
    'htc':       ['htc'],
    'msi':       ['msi'],
    'canon':     ['كانون', 'canon'],
}

# Flat set of all variants (already in normalized/lowercase form)
allowed_brands = set(v.lower() for variants in brand_map.values() for v in variants)


def extract_brands(text):
    """Return list of canonical brand names found in text."""
    found = set()
    text_lower = text.lower()  # text is already normalized at this point
    for brand, variants in brand_map.items():
        for v in variants:
            if ' ' in v:
                if v in text_lower:
                    found.add(brand)
            else:
                if re.search(r'\b' + re.escape(v) + r'\b', text_lower):
                    found.add(brand)
    return list(found)


def count_brands(text):
    """Return number of distinct brands mentioned in the text."""
    return len(extract_brands(text))

## 6. Entity Processing & Comparison Detection

In [ ]:
# Comparison keywords — written in normalized form (post Alef/Ta Marbuta normalization)
comparison_words = [
    'افضل من', 'احسن من', 'اقوي من', 'اقوى من',
    'ارخص من', 'اغلي من', 'اغلى من', 'اسرع من',
    'اجود من', 'افخم من', 'اكبر من', 'اصغر من',
    'مقارنه', 'مقارنه بين', 'فرق بين',
    'يفضل علي', 'يتفوق علي', 'يتفوق عن',
]

def has_comparison(text):
    """Return True if the text contains at least one comparison keyword."""
    return any(word in text for word in comparison_words)


def clean_entities(row):
    """Deduplicate and clean EN1/EN2 entity values from the annotation columns."""
    ents = []
    for e in [row['EN1'], row['EN2']]:
        if pd.notna(e):
            e = str(e).strip()
            if e and e.lower() not in ('none', '') and e not in ents:
                ents.append(e)
    return ents

## 7. Clean Each Dataset Separately

In [ ]:
# ── Non-Comparative Data ────────────────────────────────────────────

# Drop columns not useful for modeling
Non_df.drop(['Tweet Date', 'USER Location'], axis=1, inplace=True)
Non_df.rename(columns={'Tweet Text': 'Text'}, inplace=True)

# Apply full preprocessing pipeline
Non_df['Text'] = Non_df['Text'].apply(preprocess)

# Compute diagnostic features AFTER preprocessing
Non_df['brand_count']    = Non_df['Text'].apply(count_brands)
Non_df['has_comparison'] = Non_df['Text'].apply(has_comparison)

# NOTE: we do NOT filter rows based on comparison keywords here.
# Removing non-comparative tweets that contain comparison keywords
# creates a perfectly separable dataset and causes F1 = 1.0 (data leakage).

Non_df['Compartive'] = 'non'
Non_df['label']      = 0

print(f'Non-comparative rows: {len(Non_df)}')


In [ ]:
# ── Comparative Data ────────────────────────────────────────────────

# Drop administrative columns that don't carry modeling signal
Com_df.drop(['Date', 'Tweet_id', 'idx'], axis=1, inplace=True)

# Apply preprocessing; fillna before to avoid errors on empty cells
Com_df['Text'] = Com_df['Text'].fillna('').apply(preprocess)
Com_df['EN1']  = Com_df['EN1'].fillna('None').apply(lambda x: normalize_arabic(str(x)))
Com_df['EN2']  = Com_df['EN2'].fillna('None').apply(lambda x: normalize_arabic(str(x)))

# FIX: strip label whitespace BEFORE any comparison
Com_df['Compartive'] = Com_df['Compartive'].astype(str).str.strip().str.lower()

# FIX (CRITICAL): The xlsx file contains 504 rows labeled 'non' mixed with 2332 'com'.
# Original notebook set label=1 for ALL rows in Com_df — this wrongly labeled
# 504 non-comparative tweets as comparative (label noise → model confusion).
# Correct approach: assign label based on the actual Compartive column value.
Com_df['label'] = Com_df['Compartive'].map({'com': 1, 'non': 0})

# Drop the 2 rows where label couldn't be mapped (NaN labels after strip)
Com_df.dropna(subset=['label'], inplace=True)
Com_df['label'] = Com_df['label'].astype(int)

print(f'Com_df rows: {len(Com_df)}')
print(Com_df['label'].value_counts())

In [ ]:
# ── Extract Entity Lists ────────────────────────────────────────────

Non_df['entities'] = [[] for _ in range(len(Non_df))]
Com_df['entities'] = Com_df.apply(clean_entities, axis=1)

# ── Standardize Columns Before Merging ──────────────────────────────

cols = ['Text', 'Compartive', 'EN1', 'EN2', 'Sentiment',
        'PreferedEntity', 'label', 'entities']

# Add missing annotation columns to the non-comparative frame
for col in ['EN1', 'EN2', 'Sentiment', 'PreferedEntity']:
    Non_df[col] = 'None'

Non_df = Non_df[cols]
Com_df = Com_df[cols]

## 8. Feature Engineering

In [ ]:
# ── Merge Both Datasets ─────────────────────────────────────────────

final_df = pd.concat([Non_df, Com_df], ignore_index=True)

# Drop any rows where text is empty after preprocessing
final_df.dropna(subset=['Text'], inplace=True)
final_df['Text'] = final_df['Text'].str.strip()
final_df = final_df[final_df['Text'] != '']

# ── Compute Engineered Features ──────────────────────────────────────

final_df['brands']         = final_df['Text'].apply(extract_brands)
final_df['brand_count']    = final_df['brands'].apply(len)
final_df['has_comparison'] = final_df['Text'].apply(has_comparison)
final_df['text_length']    = final_df['Text'].apply(len)     # character count
final_df['word_count']     = final_df['Text'].apply(lambda x: len(x.split()))  # word count

print(f'Merged dataset shape: {final_df.shape}')
print(final_df['label'].value_counts())

## 9. Final Dataset Filtering

In [ ]:
# Rule 1: Comparative tweets must have at least one entity or brand mention
# NOTE: we do NOT remove non-comparative rows that contain comparison keywords.
# That filter creates a linearly separable dataset — the model just memorizes
# keywords instead of learning semantics, leading to F1 = 1.0 on test set.
final_df = final_df[
    (final_df['label'] == 0) |
    (
        (final_df['label'] == 1) &
        (
            (final_df['entities'].apply(len) >= 1) |  # annotated entity exists
            (final_df['brand_count'] >= 1) |          # brand found in text
            (final_df['has_comparison'])              # at least a comparison keyword
        )
    )
]

# Rule 2: Length constraints
final_df = final_df[
    (final_df['word_count'] >= 3) &
    (final_df['word_count'] <= 50)
]

# Rule 3: Remove exact duplicate tweets
final_df.drop_duplicates(subset=['Text'], inplace=True)

# Rule 4: Shuffle for unbiased train/val/test splitting in later phases
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset after filtering: {len(final_df)} rows')
print(final_df['label'].value_counts())
print('\nComparison keyword distribution per label:')
print(pd.crosstab(final_df['has_comparison'], final_df['label']))


## 10. Remove English Words While Preserving Brand Names

In [ ]:
# Product model qualifiers — keep these even though they're English
# because they are part of product names (S21 Ultra, iPhone Pro Max, etc.)
MODEL_QUALIFIERS = {'ultra', 'pro', 'max', 'mini', 'plus', 'lite', 'note',
                     'air', 'se', 'fe', 'neo', 'edge', 'fold', 'flip'}


def protect_brands(text):
    """
    Wrap known brand variants in __placeholders__ so they survive
    the English-stripping step.
    Sort longest first so 'iphone' is protected before 'i'.
    Multi-word brands use simple substring replace (no \\b).
    """
    for brand in sorted(allowed_brands, key=len, reverse=True):
        ph = f'__{brand.replace(" ", "_")}__'
        if ' ' in brand:
            text = text.replace(brand, ph)
        else:
            text = re.sub(r'\b' + re.escape(brand) + r'\b', ph, text)
    return text


def remove_english(text):
    """
    Remove unprotected English words EXCEPT product qualifiers.
    FIX: Original stripped 'ultra', 'pro', 'max' etc which are part of
    product names. Now they are preserved.
    FIX: Numbers are preserved (S21, PS5, i7 already protected above).
    """
    def should_remove(match):
        word = match.group(0).lower()
        # Keep model qualifiers
        if word in MODEL_QUALIFIERS:
            return word
        return ' '
    text = re.sub(r'(?<!_)\b[A-Za-z]{2,}\b(?!_)', should_remove, text)
    return text


def restore_brands(text):
    """Unwrap the __placeholder__ tokens back to original brand spellings."""
    for brand in allowed_brands:
        ph = f'__{brand.replace(" ", "_")}__'
        text = text.replace(ph, brand)
    return text


def clean_keep_brands(text):
    """Full pipeline: lowercase → protect brands → strip English → restore brands."""
    text = str(text).lower()
    text = protect_brands(text)
    text = remove_english(text)
    text = restore_brands(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


final_df['Text'] = final_df['Text'].apply(clean_keep_brands)

print('Sample after English removal:')
print(final_df['Text'].head(5).to_string())

## 11. Save Processed Dataset

In [ ]:
os.makedirs('processed', exist_ok=True)

final_df.to_csv('processed/final_dataset.csv', index=False, encoding='utf-8-sig')

print(f'Dataset saved → processed/final_dataset.csv')
print(f'Total rows  : {len(final_df)}')
print(f'Label counts:\n{final_df["label"].value_counts()}')
print(f'\nColumns: {final_df.columns.tolist()}')
print(f'\nSample:')
final_df[['Text','label','entities','brand_count','has_comparison','word_count']].head(5)